# 긴 상담의 기억과 한 번에 보는 승인

새한님, **`USE = "fake"`로 위에서부터 실행하고 Solar 셀은 건너뛰세요.** 고객은 배송을 묻고 기본 배송 메모를 남깁니다. 이야기가 쿠폰으로 넘어갔다가 다시 배송으로 돌아옵니다. 이어서 주문 하나는 취소하고 다른 하나는 주소를 바꿔 달라고 합니다. 다음 날에는 새 대화를 열어 기본 배송 메모를 묻습니다. 주문과 주소는 실습용입니다.

| 새 용어 | 여기서 뜻하는 것 |
|---|---|
| 단기 기억과 장기 기억 | 단기 기억은 한 대화의 진행을 잇습니다. 장기 기억은 같은 고객의 다음 상담에서도 쓸 사실을 담습니다. |
| 스토어(store), 네임스페이스(namespace) | 스토어는 여러 대화가 함께 쓸 저장소입니다. 네임스페이스는 고객별 저장 위치를 구분하는 경로입니다. |
| 실행 맥락(runtime context) | 실행을 시작하거나 재개할 때 앱이 넘기는 값입니다. 이 실습에서는 고객 ID를 담습니다. |
| 요약 미들웨어 | 오래된 메시지 묶음을 요약 한 줄로 바꿔 뒤의 대화를 이어 주는 단계입니다. |
| 묶음 승인(batched HITL) | 모델 응답 한 번이 낸 확인 대상들을 한 화면에 띄우고, 항목별 결정을 한 번에 받는 방식입니다. |

앞에서 다룬 내용은 [대화 기억과 승인](02_대화_기억과_사람의_승인.ipynb), [한 대화에서 반품 이어 가기](03_한_대화에서_반품_신청_이어_가기.ipynb), [담당 바꾸기와 전달 범위](04_배송_상담에서_환불_담당으로_인계.ipynb), [도구 오류와 체크포인트 분기](05_도구_오류_재시도와_체크포인트_분기.ipynb)에서 다시 볼 수 있어요.

실습 자료는 모두 커널 안에 있습니다. 처음부터 다시 보려면 커널을 재시작하고 같은 순서로 실행하세요.

## 준비

프로젝트 경로를 맞추고 현재 커널의 Python과 패키지 버전을 출력합니다. `show()`는 모델의 도구 요청과 도구 응답을 한 줄씩 보여 줍니다. 응답의 `call`은 원래의 도구 호출 ID입니다. `thread_config()`는 새 대화 이름표를 만듭니다.

In [ ]:
import json
import os
import re
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path
from threading import Lock
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

sys.dont_write_bytecode = True

REPO = Path("/home/home/projects/shopagent")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from langchain.agents import create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    SummarizationMiddleware,
    wrap_model_call,
)
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command

print("Python:", sys.version.split()[0])
for package in ("langchain", "langchain-core", "langgraph", "langchain-openai", "ipykernel"):
    print(f"{package}: {version(package)}")


def show(messages):
    for message in messages:
        line = f"{message.type}: {message.content}"
        if isinstance(message, AIMessage) and message.tool_calls:
            line += " | tool_calls=" + json.dumps(message.tool_calls, ensure_ascii=False)
        if isinstance(message, ToolMessage):
            line += f" | name={message.name} status={message.status} call={message.tool_call_id}"
        print(line.replace("\n", " ↵ "))


def thread_config(label):
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex[:8]}"}}

### Gemma 연결 준비

`LocalLLM(port=18380).client`로 연결 객체를 만들고 이 컴퓨터의 `/health` 응답을 최대 2초 동안 기다립니다. 연결되면 상태 코드가 나오고, 연결에 실패하면 가짜 모델로 계속할 수 있다는 안내가 나옵니다.

나중에 Gemma를 쓰려면 별도 터미널에서 아래 명령을 실행하세요. 끝낼 때는 그 터미널에서 **Ctrl+C**를 누르세요. 가짜 모델 실습에는 서버가 필요 없어요.

~~~bash
/home/home/llama.cpp/build/bin/llama-server -m /home/home/models/gemma-4-12b-it-Q4_K_M.gguf --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 --host 127.0.0.1 --port 18380 --n-cpu-moe 0 --reasoning off --reasoning-budget 0
~~~

In [ ]:
import httpx
from shop.agents import LocalLLM

gemma_llm = LocalLLM(port=18380).client
try:
    with httpx.Client(trust_env=False) as health_client:
        health_response = health_client.get("http://127.0.0.1:18380/health", timeout=2)
    print(f"Gemma /health 응답 상태 코드: {health_response.status_code}")
except httpx.HTTPError:
    print("Gemma 서버에 연결할 수 없어요. 가짜 모델로 계속할 수 있습니다.")

### Solar 연결 준비

가짜 모델 실습에서는 이 셀을 건너뛰세요. 앞 실습처럼 bash의 점 명령으로 키 설정을 불러옵니다. 화면에는 준비 여부만 `True` 또는 `False`로 나옵니다. `False`라면 가짜 모델을 선택하라는 안내가 이어집니다. 키, 환경 변수와 셸의 출력은 화면에 표시하지 마세요.

**Solar로 실제 모델을 호출하면 비용이 듭니다.** 요약을 만들 때도 모델을 한 번 더 호출합니다.

In [ ]:
from shop.agents import LocalLLM
from shop.servers import CONFIGS


def load_solar_key():
    try:
        result = subprocess.run(
            ["bash", "-c", 'set -a; . ./upstage.env; set +a; printf %s "$UPSTAGE_API_KEY"'],
            cwd=REPO, capture_output=True, text=True, timeout=5,
        )
        if result.stdout.strip():
            os.environ["UPSTAGE_API_KEY"] = result.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        pass
    return bool(os.environ.get("UPSTAGE_API_KEY", "").strip())


solar_loaded = load_solar_key()
print("Upstage 키 준비 여부:", solar_loaded)
solar_llm = LocalLLM(config=CONFIGS["solar-pro4"]).client if solar_loaded else None
if solar_llm is None:
    print("Solar를 준비하려면 키가 필요해요. 지금은 USE = 'fake'로 진행하세요.")

### 가짜 모델

준비 메시지가 나오면 실습할 수 있어요. `ServiceChatModel`은 `BaseChatModel`과 `bind_tools()`를 써서 정해 둔 문의에 답합니다. 메모를 읽고 쓸 때는 도구를 호출합니다. “아까 언제 출고된다고 했죠?”에는 이번에 받은 조회 결과나 요약에서만 단서를 찾습니다.

요약 미들웨어가 별도의 요청을 보내면 첫 줄로 그 요청을 알아봅니다. 원문에서 출고 예정과 쿠폰 조건을 골라 남깁니다. 물류 작업번호는 생략해, 요약에서 빠진 세부값이 어떤 질문에 영향을 주는지 볼 수 있게 했어요.

In [ ]:
SUMMARY_MARKER = "상담 기록 요약 요청"
SUMMARY_PROMPT = (
    f"{SUMMARY_MARKER}\n"
    "아래 상담 기록에서 출고 예정과 쿠폰 조건을 뽑아 짧게 정리하세요. "
    "출고 예정과 주문 번호는 함께 남기세요. 물류 작업번호와 고객의 기본 배송 메모는 생략하세요.\n\n"
    "{messages}"
)
ORDER_NUMBER = re.compile(r"\bO[0-9]+\b", re.ASCII)
DISPATCH_SENTENCE = re.compile(r"O[0-9]+ 출고 예정은 [^.\n]+입니다\.")
WAREHOUSE_SENTENCE = re.compile(r"물류 작업번호는 W-[0-9]+입니다\.")
COUPON_SENTENCE = re.compile(
    r"이번 달 쿠폰은 상품 합계 [0-9,]+원부터 쓸 수 있어요\. 다른 쿠폰과 [^.\n]+\."
)
DELIVERY_NOTES = ("문 앞", "경비실")


class ServiceChatModel(BaseChatModel):
    @property
    def _llm_type(self):
        return "service-scripted"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self

    @staticmethod
    def _calls(customer, *actions):
        return AIMessage(content="", tool_calls=[
            {"name": name, "args": args, "id": f"{customer.id}-{name}", "type": "tool_call"}
            for name, args in actions
        ])

    @staticmethod
    def _tool_reply(messages):
        request = next(message for message in reversed(messages) if isinstance(message, AIMessage))
        answers = {
            message.tool_call_id: message for message in messages if isinstance(message, ToolMessage)
        }
        replies = []
        for call in request.tool_calls:
            answer = answers[call["id"]]
            if answer.status == "error":
                action = "취소" if answer.name == "cancel_order" else "배송지 변경"
                replies.append(f"{call['args']['order_id']} 주문의 {action} 요청은 실행하지 않았어요.")
            else:
                replies.append(json.loads(answer.content)["message"])
        return " ".join(replies)

    @staticmethod
    def _recall_dispatch(messages):
        evidence = "\n".join(
            str(message.content) for message in messages
            if (isinstance(message, ToolMessage) and message.name == "check_dispatch")
            or message.additional_kwargs.get("lc_source") == "summarization"
        )
        schedule = DISPATCH_SENTENCE.search(evidence)
        warehouse = WAREHOUSE_SENTENCE.search(evidence)
        if not schedule:
            return "지금 받은 기록에는 앞선 출고 안내가 없어요. 주문을 다시 확인할 수 있어요."
        detail = warehouse.group() if warehouse else "물류 작업번호는 다시 확인해야 해요."
        return f"{schedule.group()} {detail}"

    def _reply(self, messages):
        latest = messages[-1]
        text = str(latest.content)
        if isinstance(latest, ToolMessage):
            return AIMessage(content=self._tool_reply(messages))

        if text.startswith(SUMMARY_MARKER):
            facts = [pattern.search(text) for pattern in (DISPATCH_SENTENCE, COUPON_SENTENCE)]
            summary = " ".join(fact.group() for fact in facts if fact)
            return AIMessage(content=summary or "앞서 고객의 문의를 확인했어요.")

        orders = ORDER_NUMBER.findall(text)
        if len(orders) == 2 and "취소" in text and "새 배송지:" in text:
            address = text.partition("새 배송지:")[2].strip()
            return self._calls(
                latest,
                ("cancel_order", {"order_id": orders[0]}),
                ("change_shipping_address", {"order_id": orders[1], "address": address}),
            )
        if "기본 배송 메모" in text:
            if "기억해" in text:
                note = next((value for value in DELIVERY_NOTES if value in text), None)
                if note:
                    return self._calls(latest, ("save_delivery_note", {"note": note}))
                return AIMessage(content="문 앞과 경비실 중 어디로 둘까요?")
            return self._calls(latest, ("read_delivery_note", {}))
        if "아까" in text and "출고" in text:
            return AIMessage(content=self._recall_dispatch(messages))
        if orders and "출고" in text:
            return self._calls(latest, ("check_dispatch", {"order_id": orders[0]}))
        if "쿠폰" in text and "얼마부터" in text:
            return self._calls(latest, ("get_coupon_terms", {}))
        if "운송장" in text:
            return AIMessage(content="출고 후 주문 상세에서 운송장을 확인할 수 있어요.")
        return AIMessage(content="네, 이어서 궁금한 점이 있으면 말씀해 주세요.")

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self._reply(messages)
        return ChatResult(generations=[ChatGeneration(message=reply)])


fake_llm = ServiceChatModel()
print("가짜 모델을 준비했어요.")

### 모델 선택

`USE`는 `fake`, `gemma`, `solar` 중 하나입니다. 선택한 모델 이름이 출력됩니다. 준비가 필요한 경우 선택 셀에서 안내가 나옵니다. 모델을 바꾸면 이 셀부터 다시 실행하세요. 아래 수치와 답변은 가짜 모델 기준입니다. 실제 모델은 도구 선택과 순서가 달라질 수 있어요.

In [ ]:
USE = "fake"  # "fake" | "gemma" | "solar"
models = {"fake": fake_llm, "gemma": globals().get("gemma_llm"), "solar": globals().get("solar_llm")}
assert USE in models, "USE는 fake, gemma, solar 중 하나여야 합니다."
llm = models[USE]
assert llm is not None, "해당 모델을 준비하거나 USE = 'fake'로 바꾸세요."
print("선택한 모델:", USE)

## 1. 주문 자료와 고객 선호 준비

O101은 결제완료, O102는 상품준비중입니다. 둘 다 출고 전이라 승인 뒤 변경할 수 있어요. 출고 조회와 쿠폰 조회는 자료를 읽습니다. 출고 조회에는 출고 예정과 한 번의 물류 조회에 쓰는 작업번호가 있습니다.

`cancel_order`와 `change_shipping_address`는 실행할 때 고객 소유와 출고 상태를 다시 봅니다. 승인 대기 중에 주문 상태가 바뀔 수 있기 때문이에요. 조회와 수정에 같은 잠금을 써서 주문을 읽거나 바꾸는 동안 상태가 섞이지 않게 합니다. `writes`에는 실제로 바뀐 주문만 남깁니다.

In [ ]:
class CustomerContext(TypedDict):
    customer_id: str


OrderId = Annotated[str, "작업 대상 주문 번호. 예: O101"]
BEFORE_DISPATCH = ("결제완료", "상품준비중")


def make_toy_shop():
    return {
        "orders": {
            "O101": {
                "customer_id": "C101", "status": "결제완료", "address": "서울시 성동구 새봄길 1",
                "dispatch_window": "내일 오후", "warehouse_ref": "W-19",
            },
            "O102": {
                "customer_id": "C101", "status": "상품준비중", "address": "서울시 성동구 새봄길 10",
                "dispatch_window": "내일 오전", "warehouse_ref": "W-27",
            },
        },
        "coupon": {"minimum": 30_000, "stackable": False},
        "writes": [],
    }


def make_order_tools(shop):
    orders = shop["orders"]
    writes = shop["writes"]
    lock = Lock()

    def owned_order(order_id, runtime):
        order = orders.get(order_id)
        if order and order["customer_id"] == runtime.context["customer_id"]:
            return order
        return None

    @tool
    def check_dispatch(order_id: OrderId, runtime: ToolRuntime[CustomerContext]) -> dict:
        """출고 전 주문의 출고 예정과 물류 작업번호를 조회합니다."""
        with lock:
            order = owned_order(order_id, runtime)
            if order is None or order["status"] not in BEFORE_DISPATCH:
                return {"ok": False, "message": f"{order_id} 주문의 출고 예정을 확인할 수 없어요."}
            return {"ok": True, "message": (
                f"{order_id} 출고 예정은 {order['dispatch_window']}입니다. "
                f"물류 작업번호는 {order['warehouse_ref']}입니다."
            )}

    @tool
    def get_coupon_terms() -> dict:
        """이번 달 쿠폰의 최소 주문 금액과 중복 적용 조건을 조회합니다."""
        coupon = shop["coupon"]
        stacking = "다른 쿠폰과 함께 쓸 수 있어요." if coupon["stackable"] else "다른 쿠폰과 중복 적용되지 않아요."
        return {"ok": True, "message": f"이번 달 쿠폰은 상품 합계 {coupon['minimum']:,}원부터 쓸 수 있어요. {stacking}"}

    @tool
    def cancel_order(order_id: OrderId, runtime: ToolRuntime[CustomerContext]) -> dict:
        """고객 확인 뒤 출고 전 주문을 취소합니다."""
        with lock:
            order = owned_order(order_id, runtime)
            if order is None or order["status"] not in BEFORE_DISPATCH:
                return {"ok": False, "message": f"{order_id} 주문은 지금 취소할 수 없어요."}
            order["status"] = "취소완료"
            writes.append(("cancel_order", order_id, order["status"]))
        return {"ok": True, "message": f"{order_id} 주문을 취소했어요."}

    @tool
    def change_shipping_address(
        order_id: OrderId,
        address: Annotated[str, "고객이 요청한 새 배송지 전체"],
        runtime: ToolRuntime[CustomerContext],
    ) -> dict:
        """고객 확인 뒤 출고 전 주문의 배송지를 바꿉니다."""
        with lock:
            order = owned_order(order_id, runtime)
            if order is None or order["status"] not in BEFORE_DISPATCH or not address.strip():
                return {"ok": False, "message": f"{order_id} 주문의 배송지를 변경할 수 없어요."}
            if order["address"] != address:
                order["address"] = address
                writes.append(("change_shipping_address", order_id, address))
        return {"ok": True, "address": address, "message": f"{order_id} 주문의 배송지는 {address}입니다."}

    return [check_dispatch, get_coupon_terms, cancel_order, change_shipping_address]

### 기본 배송 메모는 고객별로 저장하기

고객이 기억해 달라고 한 값을 `save_delivery_note`로 씁니다. `read_delivery_note`는 다음 상담에서 그 값을 읽습니다. 둘 다 `runtime.context["customer_id"]`로 고객을 찾고 `runtime.store`를 씁니다. 예를 들어 C101의 메모는 `("customer_preferences", "C101")` 아래 `"delivery_note"`에 저장됩니다. `get()`이 준 항목의 `value`에서 실제 값을 꺼냅니다.

앱이 확인한 고객 ID를 `context`에 넣는 구조예요. 모델은 메모 선택지만 보냅니다. 실제 서비스에서는 로그인한 고객 ID로 이 경로를 만들고, 고객이 허락한 선호만 남기세요. 예시는 ‘문 앞’과 ‘경비실’로 범위를 좁혔습니다. 카드 번호, 인증정보, 주소 전체와 상담 원문은 장기 기억에서 제외하세요.

`InMemoryStore`는 같은 커널 안에서 대화를 넘어 값을 유지합니다. 이 실습에서 ‘다음 날’에는 같은 커널을 씁니다. 커널을 재시작하면 값이 사라져요.

In [ ]:
DeliveryNote = Literal["문 앞", "경비실"]


def preference_namespace(customer_id):
    return ("customer_preferences", customer_id)


@tool
def save_delivery_note(note: DeliveryNote, runtime: ToolRuntime[CustomerContext]) -> dict:
    """고객이 앞으로도 기억해 달라고 한 기본 배송 메모를 저장합니다."""
    namespace = preference_namespace(runtime.context["customer_id"])
    runtime.store.put(namespace, "delivery_note", {"note": note})
    return {"note": note, "message": f"기본 배송 메모로 '{note}'을 저장했어요."}


@tool
def read_delivery_note(runtime: ToolRuntime[CustomerContext]) -> dict:
    """현재 고객이 저장한 기본 배송 메모를 조회합니다."""
    namespace = preference_namespace(runtime.context["customer_id"])
    item = runtime.store.get(namespace, "delivery_note")
    if item is None:
        return {"note": None, "message": "아직 저장한 기본 배송 메모가 없어요."}
    note = item.value["note"]
    return {"note": note, "message": f"기본 배송 메모는 '{note}'이에요."}

## 2. 같은 상담을 짧게 보내는 두 방법

`create_agent`에 `checkpointer=InMemorySaver()`와 `store=store`를 함께 넘깁니다. 체크포인터는 `thread_id`별 메시지와 대기 중인 실행을 보관합니다. 스토어는 고객별 선호를 보관합니다. 1번 상담과 다음 날 상담은 각각 다른 체크포인트에서 시작하고 같은 스토어를 써요.

- **트리밍**에서는 `wrap_model_call`에서 최근 고객 턴부터 잘라 `request.override(messages=...)`로 보냅니다. 사람 메시지에서만 자르면 모델의 도구 요청과 그 결과가 함께 들어갑니다. 계속 진행할 상태에는 원문이 쌓입니다.
- **요약**에서는 설치된 LangChain 1.4.7의 `SummarizationMiddleware(model, trigger=..., keep=...)`를 씁니다. `trigger=("messages", 11)`은 호출 직전 대화 메시지가 11개 이상일 때 요약을 시도합니다. `keep=("messages", 4)`는 최근 4개를 남기는 목표입니다. 도구 호출과 결과를 함께 남기려고 경계를 옮길 수 있어요. 새 상태에는 최근 기록과 요약 메시지가 남습니다. 이 요약은 고객의 장기 기억과 별개예요. 과거 체크포인트는 여전히 당시 원문을 담을 수 있습니다.

`summary_prompt`의 `{messages}` 자리에 요약할 앞부분을 넣습니다. 실습 입력은 짧아서 `trim_tokens_to_summarize=None`으로 요약 모델에 그 앞부분을 그대로 줍니다. 일반 상담 요청을 보낼 때는 `model_inputs`에 호출 직전 상태의 길이와 그 요청의 실제 메시지를 적어 둡니다. 시스템 지시도 모델 입력에 포함됩니다.

`HumanInTheLoopMiddleware`는 이 두 에이전트에 똑같이 붙입니다. 설치된 버전은 `interrupt_mode="batched"`를 지원합니다. 바꾸는 도구 두 개가 확인 대상이에요. 이 모드의 중단 값은 3번에서 읽습니다.

In [ ]:
SERVICE_PROMPT = (
    "배송과 쿠폰 상담을 돕습니다. 고객에게 한국어로 짧게 답하세요. "
    "주문 번호가 있는 새 출고 조회에는 check_dispatch를, 쿠폰 조건 조회에는 get_coupon_terms를 쓰세요. "
    "출고 예정이나 작업번호를 다시 물으면 받은 기록의 단서를 쓰고, 빠진 세부값은 다시 확인이 필요하다고 답하세요. "
    "고객이 기본 배송 메모를 기억해 달라고 하면 save_delivery_note를, 저장한 메모를 물으면 read_delivery_note를 쓰세요. "
    "한 고객 말에서 한 주문의 취소와 다른 주문의 배송지 변경을 함께 요청하면 "
    "cancel_order와 change_shipping_address를 같은 응답에 요청하세요. 확인은 프로그램이 받습니다. "
    "도구 결과를 받으면 그 결과를 안내하세요. 거절된 작업이나 수정 전 요청을 다시 실행하지 마세요."
)


def describe_cancel(call, state, runtime):
    return f"{call['args']['order_id']} 주문을 취소할까요?"


def describe_address(call, state, runtime):
    args = call["args"]
    return f"{args['order_id']} 배송지를 '{args['address']}'로 바꿀까요?"


def make_session_agent(model, shop, store, model_inputs, *, trim_turns=None, summary_trigger=None, summary_keep=4):
    @wrap_model_call
    def prepare_model_input(request, handler):
        if trim_turns is not None:
            turn_starts = [index for index, msg in enumerate(request.messages) if isinstance(msg, HumanMessage)]
            if len(turn_starts) > trim_turns:
                request = request.override(messages=request.messages[turn_starts[-trim_turns]:])

        sent = [request.system_message, *request.messages]
        model_inputs.append((len(request.state["messages"]), sent))
        return handler(request)

    middleware = [prepare_model_input]
    if summary_trigger is not None:
        middleware.append(SummarizationMiddleware(
            model=model,
            trigger=("messages", summary_trigger),
            keep=("messages", summary_keep),
            summary_prompt=SUMMARY_PROMPT,
            trim_tokens_to_summarize=None,
        ))
    middleware.append(HumanInTheLoopMiddleware(
        interrupt_mode="batched",
        interrupt_on={
            "cancel_order": {"allowed_decisions": ["approve", "reject"], "description": describe_cancel},
            "change_shipping_address": {
                "allowed_decisions": ["approve", "edit", "reject"],
                "description": describe_address,
            },
        },
        edit_notice=None,
    ))
    return create_agent(
        model,
        [*make_order_tools(shop), save_delivery_note, read_delivery_note],
        system_prompt=SERVICE_PROMPT,
        context_schema=CustomerContext,
        middleware=middleware,
        checkpointer=InMemorySaver(),
        store=store,
    )


def send_turn(agent, config, context, turn_id, text):
    if agent.get_state(config).interrupts:
        raise RuntimeError("이 대화의 승인 대기에 먼저 답해 주세요.")
    agent.invoke({"messages": [HumanMessage(id=turn_id, content=text)]}, config, context=context)
    return agent.get_state(config)

### 배송에서 쿠폰으로, 그리고 다시 배송으로

두 번 모두 아래 여덟 고객 발화를 보냅니다. 각 비교는 주문, 체크포인터, 스토어를 새로 만들어요. 배송 조회, 선호 저장, 쿠폰 전환 뒤에는 모델의 답도 바로 읽을 수 있어요. 뒤의 승인과 다음 날 상담은 요약을 쓴 쪽에서 이어 갑니다.

- **최신 상태**는 그 턴이 끝난 뒤 체크포인트에 남은 메시지 수입니다. **호출 (상태, 입력)**은 일반 상담 모델을 부를 때마다 찍은 숫자예요. 왼쪽은 바로 그때의 상태, 오른쪽은 시스템 지시 한 개를 포함한 실제 입력입니다. 요약을 만드는 별도 요청은 여기서 세지 않습니다. 도구 결과를 받은 뒤 모델에 다시 물으면 한 턴에 숫자 쌍이 두 개 나옵니다.
- 기본값에서 마지막 질문이 끝난 뒤 최신 상태는 트리밍 22개, 요약 10개입니다. 바로 앞 모델 입력은 각각 4개와 10개예요.
- 마지막 **배송 조회 원문 (상태, 입력)**은 첫 조회의 `ToolMessage`가 최신 상태와 마지막 모델 입력에 있는지 보여 줍니다. 트리밍에서는 `(True, False)`입니다. 요약에서는 `(False, False)`입니다. 그 대신 요약 한 건이 남고, 영어 접두사 다음에 출고 예정이 보입니다. 가짜 모델의 마지막 답변에서 출고 예정과 물류 작업번호 중 무엇을 말할 수 있는지 비교하세요.
- 실제 모델이 배송이나 쿠폰 문의 중에 변경을 요청하면 승인 대기를 알리고 그 비교를 거기서 멈춥니다.

In [ ]:
TRIM_TURNS = 2
SUMMARY_TRIGGER = 11
SUMMARY_KEEP = 4
customer = {"customer_id": "C101"}

day_one_turns = [
    ("배송 조회", "O102 주문은 언제 출고돼요? 물류 작업번호도 알려 주세요."),
    ("메모 저장", "앞으로 기본 배송 메모는 문 앞으로 기억해 주세요."),
    ("운송장", "출고되면 운송장은 어디에서 봐요?"),
    ("쿠폰으로 전환", "배송은 잠깐 됐고요. 이번 달 쿠폰은 얼마부터 쓸 수 있어요?"),
    ("쿠폰 후속", "다른 쿠폰과 중복 적용은 안 되는군요. 이번 쿠폰으로 볼게요."),
    ("장바구니", "장바구니 금액도 확인했어요. 결제는 나중에 할게요."),
    ("쿠폰 마무리", "쿠폰 설명은 여기까지면 충분해요."),
    ("배송 다시 질문", "아까 O102 출고가 언제라고 했죠? 물류 작업번호도 다시 알려 주세요."),
]


def has_dispatch_result(messages):
    return any(isinstance(msg, ToolMessage) and msg.name == "check_dispatch" for msg in messages)


def run_day_one(agent, config, model_inputs):
    for number, (topic, text) in enumerate(day_one_turns, start=1):
        offset = len(model_inputs)
        snapshot = send_turn(agent, config, customer, f"day-one-{number}", text)
        messages = snapshot.values["messages"]
        counts = [(count, len(sent)) for count, sent in model_inputs[offset:]]
        print(f"{number}. {topic} | 최신 상태 {len(messages)} | 호출 (상태, 입력) {counts}")
        if number in (1, 2, 4):
            show(messages[-1:])
        if snapshot.interrupts:
            print("승인 대기가 생겨 여기서 상담 비교를 멈춥니다.")
            break

    print("배송 조회 원문 (상태, 입력):", (
        has_dispatch_result(messages), has_dispatch_result(model_inputs[-1][1])
    ))
    summaries = [msg for msg in messages if msg.additional_kwargs.get("lc_source") == "summarization"]
    print("최신 상태의 요약 수:", len(summaries))
    show(summaries)
    print("마지막 모델 응답")
    show(messages[-1:])


trim_inputs, summary_inputs = [], []
trim_shop, summary_shop = make_toy_shop(), make_toy_shop()
trim_store, summary_store = InMemoryStore(), InMemoryStore()
trim_config, summary_config = thread_config("trim"), thread_config("summary")
trim_agent = make_session_agent(llm, trim_shop, trim_store, trim_inputs, trim_turns=TRIM_TURNS)
summary_agent = make_session_agent(
    llm, summary_shop, summary_store, summary_inputs,
    summary_trigger=SUMMARY_TRIGGER, summary_keep=SUMMARY_KEEP,
)

print("최근 턴만 보내기")
run_day_one(trim_agent, trim_config, trim_inputs)
print()
print("앞부분을 요약해서 보내기")
run_day_one(summary_agent, summary_config, summary_inputs)

## 3. 취소와 배송지 변경을 한 화면에서 확인하기

같은 대화에서 고객이 O101은 취소하고 O102는 배송지를 바꿔 달라고 합니다. `ai` 줄에서 한 응답에 담긴 두 도구 호출과 순서를 보세요. 대기 중단 수는 **1**입니다. 그 하나의 `interrupt.value`에 목록 두 개가 있어요.

- `action_requests`는 확인할 작업입니다. 각 항목에 도구 이름과 인자, 화면에 보여 줄 문구가 있습니다.
- `review_configs`는 각 작업에서 고를 수 있는 결정입니다. 이 실습에서는 배송지 변경에만 `edit`을 열었습니다.

이 시점의 O101은 결제완료, O102의 배송지는 새봄길 10입니다. 주문 변경 기록은 비어 있어요. 실제 모델에서는 도구 호출과 중단 수가 달라질 수 있어요. 앞선 승인 대기가 있으면 그 대기를 먼저 보여 줍니다.

In [ ]:
batch_request = "O101은 취소하고 O102는 배송지를 바꿔 주세요. 새 배송지: 서울시 성동구 가온길 20"


def show_orders(shop):
    print("O101 상태:", shop["orders"]["O101"]["status"])
    print("O102 배송지:", shop["orders"]["O102"]["address"])
    print("주문 변경 기록:", sorted(shop["writes"], key=lambda entry: entry[1]))


batch_pause = summary_agent.get_state(summary_config)
if batch_pause.interrupts:
    print("앞서 생긴 승인 대기를 표시합니다.")
else:
    batch_pause = send_turn(summary_agent, summary_config, customer, "batch-request", batch_request)
show(batch_pause.values["messages"][-1:])
print("대기 중단 수:", len(batch_pause.interrupts))
for pending in batch_pause.interrupts:
    print("interrupt.value:", json.dumps(pending.value, ensure_ascii=False, indent=2))
show_orders(summary_shop)

### 취소는 승인하고 배송지는 고쳐서 실행하기

한 중단 ID에 `{"decisions": [...]}`를 보냅니다. 목록의 각 위치는 `action_requests`의 같은 위치에 답합니다. `resume_batch()`는 확인한 도구와 주문 번호로 결정을 고른 뒤 그 순서대로 목록을 만듭니다. 기본 실행에서 제출하는 결정은 첫째 `approve`, 둘째 `edit`입니다. 예정된 두 작업으로 구성된 중단을 찾지 못하면 주문을 건드리기 전에 안내합니다.

이 버전은 수정 전 모델 요청을 메시지에 남기고, 실행할 때 수정된 인자를 씁니다. `edit_notice=None`으로 기본 수정 안내를 끈 예시입니다. 같은 `call` ID의 요청과 도구 응답을 비교하세요. **실행된 배송지는 도구 결과의 `address`**에서 읽을 수 있습니다.

도구 응답 두 개와 모델의 마지막 안내가 보입니다. 남은 중단 수는 0이고, O101은 취소완료, O102의 배송지는 해솔길 30입니다. 주문 변경 기록은 주문 번호 순으로 보여 줍니다. 두 작업은 각자 조건을 검사해 실행합니다. 업무 성공 여부는 응답의 `ok`로 확인하세요.

In [ ]:
def resume_batch(agent, config, context, decisions_by_action):
    before = agent.get_state(config)
    if len(before.interrupts) != 1:
        print("재개할 배치 중단 한 건을 찾지 못했어요.")
        return before

    pending = before.interrupts[0]
    actions = [(item["name"], item["args"].get("order_id")) for item in pending.value["action_requests"]]
    if len(actions) != len(decisions_by_action) or set(actions) != set(decisions_by_action):
        print("확인한 작업과 대기 중인 작업이 달라요. 중단 내용을 다시 확인해 주세요.")
        return before

    decisions = [decisions_by_action[action] for action in actions]
    print("제출할 decisions:", json.dumps(decisions, ensure_ascii=False))
    original_request = before.values["messages"][-1]
    call_ids = {call["id"] for call in original_request.tool_calls}
    agent.invoke(Command(resume={pending.id: {"decisions": decisions}}), config, context=context)

    after = agent.get_state(config)
    messages = after.values["messages"]
    show([msg for msg in messages if isinstance(msg, ToolMessage) and msg.tool_call_id in call_ids])
    show(messages[-1:])
    print("남은 중단 수:", len(after.interrupts))
    return after


edited_address = "서울시 성동구 해솔길 30"
approve_and_edit = {
    ("cancel_order", "O101"): {"type": "approve"},
    ("change_shipping_address", "O102"): {
        "type": "edit",
        "edited_action": {"name": "change_shipping_address", "args": {"order_id": "O102", "address": edited_address}},
    },
}
edited_state = resume_batch(summary_agent, summary_config, customer, approve_and_edit)
show_orders(summary_shop)

### 같은 두 요청에서 취소만 거절하기

별도의 주문 자료와 대화에 같은 요청을 보냅니다. 이번 결정은 `reject`, `approve` 순입니다. 취소 쪽의 `ToolMessage`에는 영어 거절 안내와 사람이 적은 이유가 붙어요. 이때 `status=error`는 거절을 표시합니다.

남은 중단은 0건입니다. O101은 결제완료에 머물고, O102는 고객이 처음 제시한 가온길 20으로 바뀝니다. 주문 변경 기록에는 배송지 변경 한 건만 남습니다.

In [ ]:
reject_shop = make_toy_shop()
reject_agent = make_session_agent(llm, reject_shop, InMemoryStore(), [])
reject_config = thread_config("reject")
send_turn(reject_agent, reject_config, customer, "batch-request", batch_request)

reject_and_approve = {
    ("cancel_order", "O101"): {"type": "reject", "message": "O101은 그대로 받을게요."},
    ("change_shipping_address", "O102"): {"type": "approve"},
}
rejected_state = resume_batch(reject_agent, reject_config, customer, reject_and_approve)
show_orders(reject_shop)

## 4. 다음 날, 메시지는 0개여도 메모는 남아요

요약을 썼던 에이전트에서 새 `thread_id`를 만듭니다. 첫 질문 전 메시지 수는 **0**입니다. 스토어는 어제와 같고 `context`에는 같은 고객 C101을 넘깁니다.

이번 대화에는 새 질문, `read_delivery_note` 호출, 도구 결과, 답변만 표시됩니다. `tool_calls`에서 `args={}`인지 보세요. 도구가 실행 맥락의 고객 ID로 저장 위치를 찾아 ‘문 앞’을 꺼냅니다. 이 메모를 남긴 어제의 문장을 새 대화에 복사할 필요가 없어요.

In [ ]:
next_config = thread_config("next-day")
next_customer = customer
empty_state = summary_agent.get_state(next_config)
print("첫 질문 전 메시지 수:", len(empty_state.values.get("messages", [])))
next_day = send_turn(
    summary_agent, next_config, next_customer, "next-day-note", "제 기본 배송 메모를 알려 주세요."
)
show(next_day.values["messages"])

## 5. 서비스를 만들 때 고를 것

| 어디에 둘까 | 이 상담에서 맡기는 일 |
|---|---|
| 대화 상태와 체크포인터 | 메시지와 대기 중인 승인처럼 이 상담을 이어 가는 데 필요한 값입니다. 승인할 주문과 인자가 들어간 대기 상태는 재개할 때 다시 읽습니다. |
| 고객별 스토어 | 다음 방문에도 쓸 선호를 작은 필드로 저장합니다. 고객이 고치거나 지울 수 있게 보관 범위를 정하세요. |
| 주문 DB 역할의 딕셔너리 | 현재 주문 상태와 배송지의 기준입니다. 승인 직후에도 도구가 여기서 조건을 다시 읽습니다. 쿠폰 조건도 업무 자료에서 읽습니다. |

최근 몇 턴이면 충분하고 원문이 따로 필요할 때는 **트리밍**이 간단해요. 입력 비용을 줄이지만 이전 화제로 돌아오면 필요한 단서가 빠질 수 있고, 최신 체크포인트에는 원문이 계속 쌓입니다. 여러 화제를 이어 가야 할 때는 **요약**으로 중요한 단서를 남길 수 있어요. 요약 호출의 시간과 비용이 추가되고, 이번 작업번호처럼 세부값이 빠질 수 있습니다. 주문 상태처럼 정확해야 하는 값은 조회 도구로 다시 확인하세요. 메시지 개수는 실습에서 세기 쉽습니다. 운영에서는 메시지마다 길이가 달라 토큰 예산도 함께 봅니다.

작업들이 한 화면에서 비교하기 좋을 때는 **묶음 승인**을 써 보세요. 각 줄에 주문과 바뀔 값을 보여 주세요. 모든 줄의 결정을 받은 뒤 재개합니다. 항목을 화면에서 재정렬하더라도 응답은 원래 `action_requests`의 순서로 돌려보내야 해요. **건별 승인**은 도착한 작업부터 독립적으로 판단하기 좋아요. 여러 작업이면 각각 승인 상태를 표시하고 각 중단 ID에 답해야 합니다. 두 주문을 한 화면에서 승인해도 각 도구는 따로 실행됩니다. 전부 성공해야 하는 업무라면 주문 DB에서 별도로 실행 단위를 설계하세요.

## 직접 바꿔 볼 것

`USE = "fake"`로 하나씩 해 보세요. 각 실험이 끝나면 코드를 되돌리세요. 매번 `day-one`부터 실행하면 두 대화와 주문 자료를 새로 준비할 수 있습니다.

1. **트리밍 범위 넓히기.** `day-one`의 `TRIM_TURNS`를 `8`로 바꾸고 그 셀을 실행하세요. 트리밍 쪽 최신 상태의 크기는 같고, 마지막 입력에 배송 조회 원문이 들어옵니다. 답변에도 출고 예정과 W-27이 함께 돌아옵니다.
2. **요약 시작을 미루기.** `day-one`의 `SUMMARY_TRIGGER`를 `99`로 바꾸고 그 셀을 실행하세요. 요약 쪽 최신 상태의 요약 수가 0이 되고, 첫 배송 조회 원문과 W-27이 답변의 근거로 돌아옵니다.
3. **요약할 때 더 많이 줄이기.** `day-one`의 `SUMMARY_KEEP`를 `2`로 바꾸고 그 셀을 실행하세요. 쿠폰으로 전환하는 턴의 요약 쪽에서 첫 상담 입력이 6개에서 4개로 줄어듭니다. 쿠폰 도구 결과를 받은 뒤에는 모델이 한 번 더 호출되고, 마지막 질문에는 출고 예정만 돌아옵니다.
4. **새 선호 저장하기.** `day-one`의 ‘메모 저장’ 발화에서 ‘문 앞으로’를 ‘경비실로’로 바꾸고 `day-one`부터 `next-day`까지 실행하세요. 다음 날 도구 결과와 답변에 ‘경비실’이 나옵니다.
5. **다른 고객으로 다음 날 상담 열기.** `next-day`의 `next_customer = customer`를 `next_customer = {"customer_id": "C202"}`로 바꾸고 실행하세요. 첫 메시지 수는 여전히 0이고, 이번에는 저장한 메모가 없다는 답변이 나옵니다.
6. **승인 화면에서 주소 한 번 더 고치기.** `day-one`부터 `batch-pause`까지 실행하세요. `batch-resume`의 `edited_address`를 `"서울시 성동구 단풍길 40"`으로 바꾸고 실행하세요. 중단의 원래 요청은 가온길 20이고, 같은 호출 ID의 도구 응답과 주문에는 단풍길 40이 남습니다.
7. **두 작업 모두 거절하기.** `batch-reject`의 `reject_and_approve`에서 배송지 쪽 결정을 `{"type": "reject", "message": "주소도 그대로 둘게요."}`로 바꾸고 실행하세요. 거절 도구 응답은 두 건, 주문 변경 기록은 빈 목록입니다.
8. **승인을 기다리는 동안 출고시키기.** `day-one`부터 `batch-pause`까지 실행하세요. `batch-resume`에서 `edited_state = resume_batch(...)` 바로 앞에 `summary_shop["orders"]["O102"]["status"] = "배송중"`을 추가하고 실행하세요. O101 취소는 성공하고, O102 도구는 응답을 돌려주지만 본문의 `ok`는 `false`입니다. O102 주소는 새봄길 10에 머물고, 기록에는 O101 취소 한 건만 남습니다.

### shopagent 코드에서 이어 읽기

| 실습에서 본 것 | 프로젝트에서 찾을 곳 |
|---|---|
| 한 대화의 단기 기억 | [shop/graph_v2.py](../shop/graph_v2.py)의 `State`, `ScenarioGraph.__init__`, `build_graph`. 바깥 그래프가 `InMemorySaver`와 대화 이름표를 써서 메시지와 담당 등을 이어 갑니다. 이 경로에 고객별 LangGraph 스토어 연결은 없습니다. |
| 담당 모델에 보낼 기록 | [shop/graph_v2.py](../shop/graph_v2.py)의 `build_graph` 안 `specialist`가 공유 메시지를 담당 에이전트에 넘깁니다. `build_agent`에는 이 실습의 트리밍과 요약 미들웨어가 없습니다. [shop/agents.py](../shop/agents.py)의 `Agents.supervise`에서 별도 판정 백엔드에 주는 텍스트는 최근 메시지 10개로 제한합니다. |
| 건별 승인과 재개 | [shop/graph_v2.py](../shop/graph_v2.py)의 `build_agent`는 `per_call`로 중단합니다. `TurnMiddleware._tool_call`은 같은 모델 응답에서 확인 대상 중 첫 건만 허용합니다. `ScenarioGraph.resume_turn`은 중단 ID에 `approve`, `reject`, `respond` 중 한 결정을 보냅니다. |
| 소유권과 주문의 실제 변경 | [shop/tools.py](../shop/tools.py)의 `Shop._owned`가 주문 소유를 확인합니다. `Shop.cancel_order`와 `Shop.change_shipping_address`는 `Shop.db`의 주문 자료를 변경하고, `Shop.invoke`와 `Shop.record_call`이 실행 내역을 남깁니다. |
| 모델 연결 | [shop/agents.py](../shop/agents.py)의 `LocalLLM`과 [shop/servers.py](../shop/servers.py)의 `CONFIGS`에서 Gemma 연결과 `solar-pro4` 설정을 찾을 수 있어요. 프로젝트의 `LocalLLM.call`은 `parallel_tool_calls=False`를 줍니다. |